In [1]:
import pandas as pd

file_path = "../data/raw/Online_Retail.xlsx"

df = pd.read_excel(file_path)

df.shape

(541909, 8)

In [2]:
duplicate_count = df.duplicated().sum()

duplicate_summary = pd.DataFrame({
    "Metric": ["Total Rows", "Duplicate Rows", "Duplicate Percentage"],
    "Value": [
        len(df),
        duplicate_count,
        round((duplicate_count / len(df)) * 100, 2)
    ]
})

duplicate_summary

,Metric,Value
0,Total Rows,541909.00
1,Duplicate Rows,5268.00
2,Duplicate Percentage,0.97


In [3]:
duplicate_rows = df[df.duplicated(keep=False)].sort_values(
    by=["InvoiceNo", "StockCode", "CustomerID"]
)

duplicate_rows.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom


In [4]:
df_clean = df.drop_duplicates().copy()

print("Original rows:", len(df))
print("Rows after removing duplicates:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

Original rows: 541909
Rows after removing duplicates: 536641
Rows removed: 5268


In [5]:
df_clean.duplicated().sum()

np.int64(0)

In [8]:
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

In [9]:
df_clean["Is_Cancelled"] = (
    df_clean["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
)

cancellation_audit = (
    df_clean.groupby("Is_Cancelled")
    .agg(Transaction_Count=("InvoiceNo", "count"),
        Total_Quantity=("Quantity", "sum"),
        Total_Revenue=("Revenue", "sum")
    )
    .reset_index()
)

cancellation_audit

,Is_Cancelled,Transaction_Count,Total_Quantity,Total_Revenue
0,False,527390,5438062,1.061999e+07
1,True,9251,-275560,-8.939797e+05


In [10]:
df_clean = df_clean[~df_clean["Is_Cancelled"]].copy()

print("Rows after removing cancellations:", len(df_clean))
print("Cancelled rows remaining:", df_clean["Is_Cancelled"].sum())

Rows after removing cancellations: 527390
Cancelled rows remaining: 0


In [11]:
invalid_quantity = (
    df_clean[df_clean["Quantity"] <= 0]
    .groupby("Quantity")
    .size()
    .reset_index(name="Row_Count")
    .sort_values("Quantity")
)

invalid_quantity


,Quantity,Row_Count
0,-9600,2
1,-9058,1
2,-5368,1
3,-4830,1
4,-3667,1
...,...,...
293,-5,46
294,-4,35
295,-3,39
296,-2,58


In [12]:
quantity_issue_summary = pd.DataFrame({
    "Issue": ["Quantity = 0", "Quantity < 0"],
    "Row_Count": [
        (df_clean["Quantity"] == 0).sum(),
        (df_clean["Quantity"] < 0).sum()
    ]
})

quantity_issue_summary["Percentage"] = (
    quantity_issue_summary["Row_Count"]
    / len(df_clean)
    * 100
).round(2)

quantity_issue_summary

,Issue,Row_Count,Percentage
0,Quantity = 0,0,0.00
1,Quantity < 0,1336,0.25


In [13]:
df_clean = df_clean[df_clean["Quantity"] > 0].copy()

print("Rows after removing negative quantities:", len(df_clean))
print("Negative quantity rows remaining:", (df_clean["Quantity"] < 0).sum())


Rows after removing negative quantities: 526054
Negative quantity rows remaining: 0


In [14]:
price_issue_summary = pd.DataFrame({
    "Issue": ["UnitPrice = 0", "UnitPrice < 0"],
    "Row_Count": [
        (df_clean["UnitPrice"] == 0).sum(),
        (df_clean["UnitPrice"] < 0).sum()
    ]
})

price_issue_summary["Percentage"] = (
    price_issue_summary["Row_Count"]
    / len(df_clean)
    * 100
).round(2)

price_issue_summary

,Issue,Row_Count,Percentage
0,UnitPrice = 0,1174,0.22
1,UnitPrice < 0,2,0.00


In [15]:
df_clean = df_clean[df_clean["UnitPrice"] > 0].copy()

print("Rows after removing invalid prices:", len(df_clean))
print("Invalid price rows remaining:", (df_clean["UnitPrice"] <= 0).sum())

Rows after removing invalid prices: 524878
Invalid price rows remaining: 0


In [16]:
cleaning_summary = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Cleaned Rows",
        "Rows Removed",
        "Remaining Rows (%)"
    ],
    "Value": [
        len(df),
        len(df_clean),
        len(df) - len(df_clean),
        round((len(df_clean) / len(df)) * 100, 2)
    ]
})

cleaning_summary

,Metric,Value
0,Original Rows,541909.00
1,Cleaned Rows,524878.00
2,Rows Removed,17031.00
3,Remaining Rows (%),96.86


In [17]:
final_quality_check = pd.DataFrame({
    "Check": [
        "Duplicate rows",
        "Cancelled transactions",
        "Quantity <= 0",
        "UnitPrice <= 0"
    ],
    "Remaining_Count": [
        df_clean.duplicated().sum(),
        df_clean["Is_Cancelled"].sum(),
        (df_clean["Quantity"] <= 0).sum(),
        (df_clean["UnitPrice"] <= 0).sum()
    ]
})

final_quality_check

,Check,Remaining_Count
0,Duplicate rows,0
1,Cancelled transactions,0
2,Quantity <= 0,0
3,UnitPrice <= 0,0


In [18]:
customer_id_check = pd.DataFrame({
    "CustomerID_Status": ["Known", "Missing"],
    "Row_Count": [
        df_clean["CustomerID"].notna().sum(),
        df_clean["CustomerID"].isna().sum()
    ]
})

customer_id_check["Percentage"] = (
    customer_id_check["Row_Count"]
    / len(df_clean)
    * 100
).round(2)

customer_id_check

,CustomerID_Status,Row_Count,Percentage
0,Known,392692,74.82
1,Missing,132186,25.18


In [19]:
df_customer = df_clean[df_clean["CustomerID"].notna()].copy()

print("Customer-level segmentation rows:", len(df_customer))
print("Missing CustomerID:", df_customer["CustomerID"].isna().sum())

Customer-level segmentation rows: 392692
Missing CustomerID: 0


In [20]:
output_path = "../data/processed/customer_transactions_clean.csv"

df_customer.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(df_customer))
print("Columns:", len(df_customer.columns))

Saved: ../data/processed/customer_transactions_clean.csv
Rows: 392692
Columns: 10


In [21]:
import os

print("File exists:", os.path.exists(output_path))

File exists: True
